# Routing — Oil & Gas Control Room Alarm Triage

## Pattern
Routing uses a **classifier LLM to determine which specialist prompt should handle an input**, then routes to that specialist. This is the right pattern when:
- Inputs arrive in a single stream but require fundamentally different handling
- Different domains need different expertise, tone, and response structure
- Misrouting carries real cost (wrong escalation path, wrong engineer paged)

## O&G Use Case: Control Room Alarm Triage
A Permian Basin operations centre receives alarms from drilling, production, pipelines, and facilities simultaneously. Each alarm type requires a completely different response:
- A wellbore integrity alarm → drilling engineer, well control procedures
- A process safety alarm → HSE team, emergency response plan
- An equipment fault → maintenance, specific troubleshooting steps
- An environmental alarm → environmental team, regulatory notification clock starts
- A pipeline alarm → pipeline integrity, PHMSA reportability assessment

Routing ensures each alarm gets the right specialist response automatically — no manual triage, no wrong engineer paged at 2am.

In [ ]:
%pip install openai -q

In [ ]:
import sys
try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    sys.path.insert(0, '/Workspace' + _nb_path.rsplit('/', 1)[0])
except Exception:
    sys.path.insert(0, '.')

from util import llm_call, extract_xml

def triage_alarm(alarm_text: str, routes: dict[str, str], verbose: bool = True) -> dict:
    """Classify an alarm and route it to the appropriate specialist prompt.

    Args:
        alarm_text: Raw alarm text as received by the control room.
        routes: Dict mapping route keys to specialist system prompts.
        verbose: Print classification reasoning.

    Returns:
        Dict with route, reasoning, and specialist response.
    """
    # Step 1: Classify
    classifier_prompt = f"""You are a control room triage system for an oil and gas operator.
Classify the alarm below into exactly one of these categories: {list(routes.keys())}

Rules:
- wellbore_integrity: Any alarm related to well control, casing pressure, tubing leaks, annular pressure, kicks, or blowout risk
- process_safety: H2S, fire, gas detector, pressure relief valve lift, overpressure, toxic release, emergency shutdown activation
- equipment_fault: Mechanical failure, instrument fault, pump/compressor/separator malfunction, motor trip, valve failure
- environmental: Spill, release to environment, pit overflow, flaring exceedance, water discharge, soil contamination
- pipeline_integrity: Pressure drop on a pipeline, suspected leak, pig tracking anomaly, CP system alarm, PHMSA reportable event

Respond in XML:
<reasoning>One sentence explaining the classification decision.</reasoning>
<route>the_category_key</route>
<urgency>CRITICAL / HIGH / MEDIUM / LOW</urgency>

Alarm:\n{alarm_text}"""

    classification = llm_call(classifier_prompt)
    reasoning = extract_xml(classification, 'reasoning')
    route_key = extract_xml(classification, 'route').strip().lower()
    urgency = extract_xml(classification, 'urgency').strip()

    if route_key not in routes:
        route_key = 'equipment_fault'  # safe fallback

    if verbose:
        print(f"  Route   : {route_key.upper()}")
        print(f"  Urgency : {urgency}")
        print(f"  Reason  : {reasoning}")

    # Step 2: Specialist response
    specialist_response = llm_call(
        prompt=alarm_text,
        system_prompt=routes[route_key]
    )

    return {
        'route': route_key,
        'urgency': urgency,
        'reasoning': reasoning,
        'response': specialist_response
    }

## Specialist Response Prompts
Each route has a different system prompt encoding the right expertise, required actions, and output structure for that alarm domain.

In [ ]:
ALARM_ROUTES = {
    "wellbore_integrity": """You are a well control specialist responding to a wellbore integrity alarm.
Your response must follow well control principles and company IWCF standards.

Structure your response as:
WELLBORE INTEGRITY ALERT
Immediate actions (first 5 minutes):
  [numbered steps — be specific, use imperative verbs]
Notify immediately:
  [who to call, in what order]
Well control risk assessment:
  [kick indicators present? shut-in recommended?]
Do NOT open choke or bleed pressure without confirming with company man.
Reference: [relevant well control procedure number if applicable]""",

    "process_safety": """You are a process safety engineer responding to a process safety alarm.
Life safety is the absolute priority. Follow the hierarchy: life safety → environment → asset.

Structure your response as:
PROCESS SAFETY ALERT
Life safety actions (IMMEDIATE):
  [muster, evacuation, headcount — specific]
Hazard isolation steps:
  [ESD activation, isolation valves, depressurisation]
Emergency contacts:
  [Internal: OIM/Supervisor | External: Emergency services if needed]
Regulatory notification required? [YES/NO — if H2S or fire, likely YES]
Reference: Emergency Response Plan Section [X]""",

    "equipment_fault": """You are a maintenance engineer responding to an equipment fault alarm.
Focus on safe isolation, root cause identification, and minimum downtime.

Structure your response as:
EQUIPMENT FAULT RESPONSE
Safe isolation steps:
  [LOTO, depressurize, isolate energy sources]
Likely root cause(s):
  [based on alarm data — be specific]
Diagnostic checks:
  [what to inspect first, what instruments to check]
Spare parts / resources needed:
  [what to mobilize]
Estimated repair time: [realistic estimate]
Production impact: [deferment, bypass options]""",

    "environmental": """You are an environmental compliance officer responding to an environmental alarm.
Regulatory notification deadlines are non-negotiable — missing them creates liability.

Structure your response as:
ENVIRONMENTAL INCIDENT RESPONSE
Immediate containment actions:
  [stop the source, contain the spread — specific steps]
Volume/extent assessment:
  [how to estimate volume or affected area]
Regulatory notification requirements:
  [SPCC, PHMSA, state agency — with notification deadlines where known]
Documentation required:
  [what to record, photograph, log]
Do NOT dilute or disturb the release area before regulators are notified.""",

    "pipeline_integrity": """You are a pipeline integrity engineer responding to a pipeline alarm.
Consider PHMSA reportability and public safety as the primary concerns.

Structure your response as:
PIPELINE INTEGRITY ALERT
Immediate actions:
  [isolate segment, reduce pressure, confirm leak vs. instrument fault]
PHMSA reportability assessment:
  [is this a reportable incident? what criteria apply?]
Leak confirmation steps:
  [patrol, atmospheric monitoring, SCADA analysis]
Public safety considerations:
  [nearby population, roads, waterways — if applicable]
Notify: [control room supervisor, integrity team, regulatory if needed]"""
}

## Mock Control Room Alarms
Eight alarms as they would arrive in a control room — raw, mixed-domain, varying urgency.

In [ ]:
INCOMING_ALARMS = [
    {
        "id": "ALM-001",
        "text": """ALARM: Annular casing pressure on Well B-07 has increased from 120 psi to 890 psi
        over the last 4 hours. Tubing pressure normal at 2,100 psi. Well currently
        producing 310 bbl/d oil, 0.8 MMscfd gas. No recent workover activity.
        Field operator reports no visible leaks at surface. Time: 02:34"""
    },
    {
        "id": "ALM-002",
        "text": """ALARM: H2S gas detector at Train 2 inlet separator reading 47 ppm.
        Alarm setpoint is 10 ppm. Wind direction SW at 8 mph toward accommodation module.
        Two personnel confirmed working in the area — location: south side of Train 2 skid.
        H2S detector at accommodation module currently reading 0 ppm. Time: 14:52"""
    },
    {
        "id": "ALM-003",
        "text": """ALARM: Export pump P-201A tripped on motor overcurrent. Pump was running at
        78% capacity, 1,450 rpm. Discharge pressure at trip: 820 psi. Standby pump P-201B
        available but not started. Oil export currently halted. Pump runtime since last
        service: 6,200 hours. Last overhaul: 18 months ago. Time: 09:15"""
    },
    {
        "id": "ALM-004",
        "text": """ALARM: Level sensor on produced water storage tank TK-05 showing overflow condition.
        Tank capacity 500 bbl, sensor reading 98% full. Secondary containment berm is intact.
        Water disposal well WD-01 currently offline for valve maintenance — estimated back
        online in 3 hours. Visual inspection confirms liquid at berm drain valve. Time: 11:20"""
    },
    {
        "id": "ALM-005",
        "text": """ALARM: SCADA showing 18 psi pressure drop on 8-inch gathering pipeline segment
        between Pad C and Central Facility over the last 45 minutes. Normal operating
        pressure 485 psi, now reading 467 psi at Pad C outlet, 412 psi at Central Facility
        inlet. No maintenance activity on this segment. Flow rate slightly decreased.
        Pipeline crosses a county road at milepost 4.2. Time: 16:05"""
    },
    {
        "id": "ALM-006",
        "text": """ALARM: Pressure relief valve PSV-142 on HP separator has lifted. Separator
        operating pressure at time of lift: 1,380 psi. PSV set point: 1,320 psi.
        Flare system receiving flow. Separator feed has been at high rates following
        Well A-14 choke increase earlier today. PSV has now reseated but separator
        pressure still at 1,290 psi — 30 psi below set point. Time: 10:48"""
    },
    {
        "id": "ALM-007",
        "text": """ALARM: Flow control valve FCV-88 on gas injection line stuck in partially open
        position — command position 0%, actual position 34%. Gas injection rate unable
        to be reduced to zero as required for planned well intervention on Well C-12
        starting at 13:00 today. Actuator power supply confirmed OK. Time: 11:55"""
    },
    {
        "id": "ALM-008",
        "text": """ALARM: Produced water pit PIT-3 level rising. Pit liner appears intact but
        overflow channel toward dry creek bed is showing flow — estimated 2-3 gal/min.
        Creek bed is normally dry, approximately 200 ft from pit. Source of overflow:
        recent heavy rain combined with pit near capacity. State water quality agency
        previously issued a notice of violation for this pit last year. Time: 07:30"""
    }
]

print(f"{len(INCOMING_ALARMS)} alarms queued for triage")

## Triage All Alarms

In [ ]:
all_results = []

for alarm in INCOMING_ALARMS:
    print(f"\n{'='*60}")
    print(f"Alarm: {alarm['id']}")
    print('='*60)
    result = triage_alarm(alarm['text'], ALARM_ROUTES)
    result['id'] = alarm['id']
    all_results.append(result)
    print(f"\n--- Specialist Response ---")
    print(result['response'])

## Triage Summary

In [ ]:
print("\nALARM TRIAGE SUMMARY")
print("="*60)
print(f"{'ID':<10} {'URGENCY':<10} {'ROUTE':<22} REASONING")
print("-"*60)
for r in all_results:
    print(f"{r['id']:<10} {r['urgency']:<10} {r['route'].upper():<22} {r['reasoning'][:55]}")

## Key Takeaways

**Why routing instead of a single general prompt?**

A generic alarm response prompt produces generic responses. The routing pattern lets each domain have its own:
- **Expertise** — well control principles vs. PHMSA reportability vs. LOTO procedures
- **Output structure** — each specialist prompt produces a domain-appropriate action format
- **Tone** — process safety responses prioritise life safety; equipment fault responses focus on minimum downtime
- **Regulatory awareness** — the environmental and pipeline specialists know about notification deadlines; the equipment specialist doesn't need to

**The classifier is the critical piece.** A misclassified alarm (e.g., routing a wellbore integrity alarm to equipment fault) could mean the wrong engineer is paged, wrong procedure followed, wrong regulatory clock started. The XML-structured classifier makes the decision transparent and auditable.

**Where to go from here:**
- Connect to a real alarm management system (OSIsoft PI, Honeywell Experion, AspenTech)
- Add a confidence threshold — low-confidence classifications escalate to human triage
- Log all classifications + responses to a Delta table for audit trail
- Add parallel processing for high-volume alarm floods (combine with parallelization pattern)